# CellFM Allen WMB-10X download

This notebook mounts Google Drive, clones the CellFM repo, installs only the Allen download dependency, and runs `cellfm.data.download` against Drive-backed storage.

The full isocortex pull is large, usually tens of GB. Start with `DOWNLOAD_MODE = "list"` or `"metadata"`, then switch to `"one_shard"` or `"all_isocortex"` only when Drive has enough free space.

In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/TingXuan-Huang/single_cell.git"
WORK_ROOT = Path("/content/cellfm_work")
REPO_DIR = WORK_ROOT / "single_cell"

DRIVE_ROOT = Path("/content/drive/MyDrive/cellfm")
ABC_ROOT = DRIVE_ROOT / "data/raw/abc"
CACHE_ROOT = DRIVE_ROOT / "data/cache/wmb_isocortex_v1"
CONFIG_OUT = DRIVE_ROOT / "configs/wmb_isocortex_colab.yaml"

for path in [WORK_ROOT, ABC_ROOT, CACHE_ROOT, CONFIG_OUT.parent]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Repo dir: {REPO_DIR}")
print(f"Allen download dir: {ABC_ROOT}")
print(f"Cache dir: {CACHE_ROOT}")

In [ ]:
def run(cmd, *, cwd=None, env=None, check=True):
    cmd = [str(x) for x in cmd]
    print("\n>>> " + " ".join(shlex.quote(x) for x in cmd))
    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result

if REPO_DIR.exists():
    run(["git", "fetch", "--all", "--prune"], cwd=REPO_DIR)
    run(["git", "pull", "--ff-only"], cwd=REPO_DIR)
else:
    run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])

run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR)

# The download wrapper only needs abc-atlas-access plus the local source tree.
# Avoid `pip install -e .[allen]` here because that also pulls the heavier scVI stack.
run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "wheel"])
run([sys.executable, "-m", "pip", "install", "-q", "abc-atlas-access"])

ENV = os.environ.copy()
ENV["PYTHONPATH"] = str(REPO_DIR / "src") + os.pathsep + ENV.get("PYTHONPATH", "")

run([sys.executable, "-c", "import cellfm, abc_atlas_access; print('imports ok')"], env=ENV)

In [ ]:
# Options: list, metadata, one_shard, all_isocortex
DOWNLOAD_MODE = "list"

# Used only when DOWNLOAD_MODE = "one_shard".
ONE_SHARD = "WMB-10Xv2:WMB-10Xv2-Isocortex-1/raw"

base_cmd = [
    sys.executable,
    "-m",
    "cellfm.data.download",
    "--out",
    ABC_ROOT,
]

if DOWNLOAD_MODE == "list":
    cmd = base_cmd + ["--list"]
elif DOWNLOAD_MODE == "metadata":
    cmd = base_cmd + ["--no-expression"]
elif DOWNLOAD_MODE == "one_shard":
    cmd = base_cmd + ["--packages", ONE_SHARD]
elif DOWNLOAD_MODE == "all_isocortex":
    cmd = base_cmd
else:
    raise ValueError(f"Unknown DOWNLOAD_MODE: {DOWNLOAD_MODE}")

run(cmd, cwd=REPO_DIR, env=ENV)

In [ ]:
manifest_path = ABC_ROOT / "download_manifest.json"

if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    print(json.dumps(manifest, indent=2)[:4000])
else:
    print(f"No manifest yet: {manifest_path}")

run(["du", "-sh", ABC_ROOT], check=False)

print("\nFirst files under the Allen cache:")
shown = 0
for path in ABC_ROOT.rglob("*"):
    if path.is_file():
        print(path.relative_to(ABC_ROOT))
        shown += 1
        if shown >= 20:
            break
if shown == 0:
    print("No files found yet.")

In [ ]:
metadata_candidates = sorted(ABC_ROOT.glob("metadata/WMB-10X/*/cell_metadata_with_cluster_annotation.csv"))

if metadata_candidates:
    metadata_csv = metadata_candidates[-1]
else:
    metadata_csv = ABC_ROOT / "metadata/WMB-10X/<release>/cell_metadata_with_cluster_annotation.csv"

input_glob = str(ABC_ROOT / "expression_matrices/WMB-10X*/*/WMB-10X*-Isocortex-*-raw.h5ad")

template = (REPO_DIR / "configs/data/wmb_isocortex.yaml").read_text()
template = template.replace(
    "/gscratch/PROJECT_GROUP/data/raw/abc/expression_matrices/WMB-10X*/*/WMB-10X*-Isocortex-*-raw.h5ad",
    input_glob,
)
template = template.replace(
    "/gscratch/PROJECT_GROUP/data/raw/abc/metadata/WMB-10X/20241115/cell_metadata_with_cluster_annotation.csv",
    str(metadata_csv),
)
template = template.replace(
    "/gscratch/PROJECT_GROUP/data/cache/wmb_isocortex_v1",
    str(CACHE_ROOT),
)

CONFIG_OUT.write_text(template)
print(CONFIG_OUT.read_text())

In [ ]:
RUN_BUILD_CACHE = False

if RUN_BUILD_CACHE:
    # Colab RAM may be too small for all real Allen shards. Run this only after
    # the download is complete and the runtime has enough memory.
    run([
        sys.executable,
        "-m",
        "scripts.build_cache",
        "--data-config",
        CONFIG_OUT,
        "--cache-dir",
        CACHE_ROOT,
    ], cwd=REPO_DIR, env=ENV)
else:
    print("Skipping cache build. Set RUN_BUILD_CACHE = True when ready.")